In [1]:
import pandas as pd
import os

def load_physionet_data(path):
    all_data = []
    for file in os.listdir(path):
        if file.endswith(".psv"):
            patient_id = file.replace(".psv", "")
            df = pd.read_csv(os.path.join(path, file), sep='|')
            df["Patient_ID"] = patient_id
            all_data.append(df)
    return pd.concat(all_data, ignore_index=True)

train_A_path = r"C:\Users\gulsh\OneDrive\Desktop\research projects\sepsis prediction\training\training_setA"
train_B_path = r"C:\Users\gulsh\OneDrive\Desktop\research projects\sepsis prediction\training\training_setB"

df_A = load_physionet_data(train_A_path)
df_B = load_physionet_data(train_B_path)

df = pd.concat([df_A, df_B], ignore_index=True)
print(df.shape)
df.head()

(1552210, 42)


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,1,0,p000001
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,2,0,p000001
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,3,0,p000001
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,4,0,p000001
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,5,0,p000001


In [ ]:
import numpy as np
import pandas as pd
import gc, warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import pennylane as qml

# GLOBAL SETTINGS
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
WINDOW = 12 # 1 - 12, 7 - 18
STRIDE = 6
POS_WEIGHT = 2.0
ALPHA = 0.2
EPOCHS = 120
BATCH_SIZE = 8

# STEP 1: CLINICAL LABELING (finds the first hour when patients hits sepsis onset_index, and then we mark 3 hours before to sepsis, and also flag the ambigious hours o to t-6)
def apply_clinical_labeling(df):
    df = df.copy()
    df["SepsisLabel_shifted"] = 0
    df["use_for_training"] = 1

    for pid, g in df.groupby("Patient_ID"):
        g = g.reset_index()
        onset_idx = g[g["SepsisLabel"] == 1].index

        for t in onset_idx:
            pos_start = max(0, t - 3)
            df.loc[g.loc[pos_start:t, "index"], "SepsisLabel_shifted"] = 1

            amb_start = max(0, t - 6)
            df.loc[g.loc[amb_start:pos_start-1, "index"], "use_for_training"] = 0

    return df

# STEP 2: PREPROCESSING
FEATURES = [
    'HR','O2Sat','Temp','SBP','MAP','DBP','Resp',
    'pH','PaCO2','BUN','Creatinine','Lactate',
    'WBC','Platelets','Age'
]

def preprocess(df):
    df = df.copy()

    # 1. Basic Imputation (Same as before)
    for c in FEATURES:
        df[c] = df.groupby("Patient_ID")[c].ffill().bfill() 
        df[c] = df[c].fillna(df.groupby("Patient_ID")[c].transform("mean")) #if everything is missing use that patients average

    
    # Shock Index: Strong indicator of occult shock
    # Formula: HR / SBP
    df["ShockIndex"] = df["HR"] / df["SBP"].replace(0, 1)  # avoid div/0
    
    # Modified Shock Index: Often more sensitive than standard SI
    # Formula: HR / MAP
    df["ModShockIndex"] = df["HR"] / df["MAP"].replace(0, 1)
    
    # Pulse Pressure: Narrowing PP is an early sign of shock
    # Formula: SBP - DBP
    df["PulsePressure"] = df["SBP"] - df["DBP"]
    
    # 3. Rate of Change (The "Delta" features)
    # We now track the DELTA of the Shock Index too!
    new_features = ["HR", "MAP", "Resp", "ShockIndex", "ModShockIndex"]
    
    for c in new_features:
        # how much of the values changed compared to 6 hrs ago matches stride 
        df[f"{c}_diff6"] = df.groupby("Patient_ID")[c].diff(6).fillna(0)

    return df

# UPDATE YOUR FEATURE LIST TO INCLUDE THESE NEW ONES
ALL_FEATURES = FEATURES + [
    "HR_diff6", "MAP_diff6", "Resp_diff6", 
    "ShockIndex", "ModShockIndex", "ShockIndex_diff6", "ModShockIndex_diff6"
]

# STEP 3: WINDOWING
def make_windows(df, balance=False, max_per_patient=16, min_signal=0.15): #max_per_patient = how many windows one patient can ontribute 
    X_pos, X_neg = [], []

    for pid, g in df.groupby("Patient_ID"):
        g = g[g["use_for_training"] == 1].reset_index(drop=True)
        if len(g) < WINDOW:
            continue

        n = min((len(g)-WINDOW)//STRIDE + 1, max_per_patient)
        collected = 0

        for i in range(0, len(g)-WINDOW+1, STRIDE):
            if collected >= n:
                break

            x = g.iloc[i:i+WINDOW][ALL_FEATURES].values.astype(np.float32)
            y = g.iloc[i:i+WINDOW]["SepsisLabel_shifted"].max()

            if balance: #signal
                signal = np.abs(x[-1] - x[-7]).mean()
                if signal < min_signal:
                    continue

            (X_pos if y == 1 else X_neg).append(x)
            collected += 1

    if balance:
        n = min(len(X_pos), len(X_neg))
        X_pos, X_neg = X_pos[:n], X_neg[:n]

    X = np.array(X_pos + X_neg)
    y = np.array([1]*len(X_pos) + [0]*len(X_neg))
    return X, y

# STEP 4: MODEL
N_QUBITS = 4
QLAYERS = 1

dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    inputs = torch.clamp(inputs, -np.pi, np.pi)
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS))
    qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

qlayer = qml.qnn.TorchLayer(
    quantum_circuit,
    {"weights": (QLAYERS, N_QUBITS, 3)}
)

class QuantumSepsis(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.adapter = nn.Linear(2*n_features, N_QUBITS)
        self.q = qlayer
        self.cls = nn.Linear(N_QUBITS, 1)

    def forward(self, x):
        x_now = x[:, -1, :]
        x_prev = x[:, -7, :]

        z = torch.cat([x_now, x_now - x_prev], dim=1)

        z = self.adapter(z)
        z = torch.clamp(z, -3.0, 3.0)     # 🔒 HARD CLAMP (important)

        q = self.q(z)

        # 🔥 CRITICAL: sanitize quantum output
        q = torch.nan_to_num(q, nan=0.0, posinf=1.0, neginf=-1.0)

        logits = self.cls(q)
        return logits.squeeze(1)


# LOSS
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction="none")
        self.gamma = gamma

    def forward(self, logits, y):
        bce = self.bce(logits, y)
        pt = torch.exp(-bce)
        return ((1-pt)**self.gamma * bce).mean()

# 5-FOLD PATIENT CV
# ========================

# ENSEMBLE EXECUTION BLOCK
from sklearn.model_selection import GroupShuffleSplit
import copy

# 1. Prepare Data
df = preprocess(apply_clinical_labeling(df))

# 2. Create Hold-Out Test Set (20%) for the Ensemble to vote on
# We cannot evaluate an ensemble without a dataset that NO model has seen.
print("✂️ Splitting Hold-Out Test Set (20%)...")
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(df, groups=df.Patient_ID))

df_dev = df.iloc[train_idx]  # Development set (used for CV training)
df_test = df.iloc[test_idx]  # Final Test set (used for Ensemble voting)

print(f"Dataset Shape: Dev {df_dev.shape}, Test {df_test.shape}")

# 3. Train 5 Models (One per fold)
gkf = GroupKFold(n_splits=5)
saved_checkpoints = [] # Store paths or dicts of trained models

for fold, (tr_idx, va_idx) in enumerate(gkf.split(df_dev, groups=df_dev.Patient_ID)):
    print(f"\n🚀 Training Ensemble Model {fold+1}/5...")

    df_tr = df_dev.iloc[tr_idx]
    # We don't strictly need validation data for training, but we keep the structure
    # df_va = df_dev.iloc[va_idx] 

    # Windowing
    Xtr, ytr = make_windows(df_tr, balance=True, max_per_patient=16)
    
    # Scaling (Must fit on specific fold training data)
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr.reshape(-1, Xtr.shape[-1])).reshape(Xtr.shape)

    # Model Setup
    model = QuantumSepsis(Xtr.shape[-1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT]).to(DEVICE))
    focal = FocalLoss()
    
    def loss_fn(logits, y):
        return ALPHA*bce(logits, y) + (1-ALPHA)*focal(logits, y)

    loader = DataLoader(
        TensorDataset(torch.tensor(Xtr), torch.tensor(ytr).float()),
        batch_size=BATCH_SIZE, shuffle=True
    )

    # Training Loop
    model.train()
    for _ in range(EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

    # SAVE THE MODEL & SCALER
    # We must save the scaler stats because each model expects data scaled differently
    checkpoint = {
        'model_state': copy.deepcopy(model.state_dict()),
        'scaler_mean': scaler.mean_,
        'scaler_scale': scaler.scale_
    }
    saved_checkpoints.append(checkpoint)
    print(f"✅ Model {fold+1} Saved.")

# ENSEMBLE VOTING (INFERENCE)
print(f"\n🔮 Running Ensemble Voting on {df_test.Patient_ID.nunique()} Test Patients...")

# Prepare Test Data (Unscaled initially)
X_test_raw, y_test = make_windows(df_test, balance=False, max_per_patient=999)

ensemble_probs = np.zeros(len(y_test))

# Iterate through all 5 saved models
for i, checkpoint in enumerate(saved_checkpoints):
    # 1. Reconstruct Scaler
    scaler = StandardScaler()
    scaler.mean_ = checkpoint['scaler_mean']
    scaler.scale_ = checkpoint['scaler_scale']
    
    # 2. Scale Test Data (Using this specific model's statistics)
    X_test_scaled = scaler.transform(X_test_raw.reshape(-1, X_test_raw.shape[-1])).reshape(X_test_raw.shape)
    
    # 3. Load Model
    model = QuantumSepsis(X_test_scaled.shape[-1]).to(DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()
    
    # 4. Predict
    with torch.no_grad():
        logits = model(torch.tensor(X_test_scaled).to(DEVICE))
        logits = torch.nan_to_num(logits, nan=0.0, posinf=10.0, neginf=-10.0)
        probs = torch.sigmoid(logits).cpu().numpy()
        
    # Add to vote sum
    ensemble_probs += probs

# Average the votes
ensemble_probs /= len(saved_checkpoints)

# FINAL METRICS
auroc = roc_auc_score(y_test, ensemble_probs)
auprc = average_precision_score(y_test, ensemble_probs)

prec, rec, f1, _ = precision_recall_fscore_support(
    y_test, ensemble_probs > 0.5, average="binary", zero_division=0
)

print("\n" + "="*40)
print(f"🌟 FINAL ENSEMBLE RESULTS (n=5 Models) 🌟")
print("="*40)
print(f"Test AUROC: {auroc:.3f}")
print(f"Test AUPRC: {auprc:.3f}")  # This is the key metric
print("="*40)

✂️ Splitting Hold-Out Test Set (20%)...
Dataset Shape: Dev (1241213, 52), Test (310997, 52)

🚀 Training Ensemble Model 1/5...
✅ Model 1 Saved.

🚀 Training Ensemble Model 2/5...
✅ Model 2 Saved.

🚀 Training Ensemble Model 3/5...
✅ Model 3 Saved.

🚀 Training Ensemble Model 4/5...
✅ Model 4 Saved.

🚀 Training Ensemble Model 5/5...
✅ Model 5 Saved.

🔮 Running Ensemble Voting on 8068 Test Patients...

🌟 FINAL ENSEMBLE RESULTS (n=5 Models) 🌟
Test AUROC: 0.991
Test AUPRC: 0.722


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    average_precision_score, 
    roc_auc_score, 
    precision_score, 
    accuracy_score,
    precision_recall_curve
)

# 1. HELPER: RECOVER PATIENT IDs
def get_patient_ids_for_windows(df, window_size=12, stride=6, max_per_patient=999, balance=False):
    """
    Regenerates the list of Patient IDs that matches the X_test/y_test arrays
    created by 'make_windows'.
    """
    patient_ids_list = []
    
    for pid, g in df.groupby("Patient_ID"):
        g = g[g["use_for_training"] == 1].reset_index(drop=True)
        if len(g) < window_size:
            continue

        n = min((len(g)-window_size)//stride + 1, max_per_patient)
        collected = 0
        
        # Temp lists for balancing logic (if enabled)
        temp_ids_pos = []
        temp_ids_neg = []

        for i in range(0, len(g)-window_size+1, stride):
            if collected >= n:
                break
            
            # Determine label for this window
            y = g.iloc[i:i+window_size]["SepsisLabel_shifted"].max()
            
            if y == 1:
                temp_ids_pos.append(pid)
            else:
                temp_ids_neg.append(pid)
            
            collected += 1
            
            # If not balancing, just add immediately
            if not balance:
                patient_ids_list.append(pid)

        # If balancing was used, mimic the slicing
        if balance:
            n_min = min(len(temp_ids_pos), len(temp_ids_neg))
            patient_ids_list.extend(temp_ids_pos[:n_min])
            patient_ids_list.extend(temp_ids_neg[:n_min])
            
    return np.array(patient_ids_list)

# 2. SMOOTHING & SCORING FUNCTION
def apply_patient_smoothing_and_score(probs, y_true, patient_ids, window_size=3):
    # Create temp DF for grouped operations
    temp_df = pd.DataFrame({
        'Patient_ID': patient_ids,
        'probs': probs
    })
    
    # Apply Rolling Mean (Grouped by Patient to prevent leakage)
    temp_df['probs_smooth'] = temp_df.groupby('Patient_ID')['probs'].transform(
        lambda x: x.rolling(window=window_size, min_periods=1).mean()
    )
    
    probs_smooth = temp_df['probs_smooth'].values
    
    # Calculate Metrics on SMOOTHED probabilities
    auroc = roc_auc_score(y_true, probs_smooth)
    auprc = average_precision_score(y_true, probs_smooth)
    
    preds_hard = (probs_smooth > 0.5).astype(int)
    precision = precision_score(y_true, preds_hard, zero_division=0)
    accuracy = accuracy_score(y_true, preds_hard)
    
    return probs_smooth, auroc, auprc, precision, accuracy

# 3. EXECUTION & COMPARISON
print("🔄 Aligning Patient IDs with Predictions...")

# Use correct parameters from your global config
test_patient_ids = get_patient_ids_for_windows(
    df_test, 
    window_size=WINDOW,   
    stride=STRIDE,        
    balance=False, 
    max_per_patient=999
)

if len(test_patient_ids) != len(ensemble_probs):
    print(f"⚠️ ERROR: Mismatch! IDs: {len(test_patient_ids)} vs Preds: {len(ensemble_probs)}")
else:
    # --- A. METRICS BEFORE SMOOTHING (RAW) ---
    raw_preds_hard = (ensemble_probs > 0.5).astype(int)
    raw_acc = accuracy_score(y_test, raw_preds_hard)
    raw_auroc = roc_auc_score(y_test, ensemble_probs)
    raw_auprc = average_precision_score(y_test, ensemble_probs)
    raw_prec = precision_score(y_test, raw_preds_hard, zero_division=0)

    # --- B. METRICS AFTER SMOOTHING ---
    probs_smooth, new_auroc, new_auprc, new_prec, new_acc = apply_patient_smoothing_and_score(
        ensemble_probs, 
        y_test, 
        test_patient_ids,
        window_size=3
    )

    # --- C. REPORT ---
    print(f"\n" + "="*50)
    print(f"📊 IMPACT OF PATIENT-LEVEL SMOOTHING")
    print("="*50)
    print(f"{'METRIC':<15} | {'BEFORE (Raw)':<15} | {'AFTER (Smooth)':<15} | {'CHANGE'}")
    print("-" * 65)
    print(f"{'Accuracy':<15} | {raw_acc:.4f}          | {new_acc:.4f}          | {(new_acc-raw_acc):+.4f}")
    print(f"{'AUROC':<15} | {raw_auroc:.4f}          | {new_auroc:.4f}          | {(new_auroc-raw_auroc):+.4f}")
    print(f"{'AUPRC':<15} | {raw_auprc:.4f}          | {new_auprc:.4f}          | {(new_auprc-raw_auprc):+.4f}")
    print(f"{'Precision':<15} | {raw_prec:.4f}          | {new_prec:.4f}          | {(new_prec-raw_prec):+.4f}")
    print("="*50)
    
    print("\n💡 Key Insight:")
    if new_auprc > raw_auprc:
        print("   Smoothing successfully reduced noise, improving the reliability (AUPRC) of alerts.")
    else:
        print("   Smoothing had little effect. The raw signals were already quite stable.")

🔄 Aligning Patient IDs with Predictions...

📊 IMPACT OF PATIENT-LEVEL SMOOTHING
METRIC          | BEFORE (Raw)    | AFTER (Smooth)  | CHANGE
-----------------------------------------------------------------
Accuracy        | 0.9407          | 0.9655          | +0.0249
AUROC           | 0.9906          | 0.9980          | +0.0074
AUPRC           | 0.7220          | 0.8527          | +0.1307
Precision       | 0.0877          | 0.1449          | +0.0573

💡 Key Insight:
   Smoothing successfully reduced noise, improving the reliability (AUPRC) of alerts.
